# NLP — Text Classification Exploratory Data Analysis (EDA)

Selamat datang di notebook EDA untuk proyek klasifikasi teks!
Notebook ini akan membantu Anda memahami dataset sebelum training model.

## Tujuan
- Memahami distribusi kelas (labels)
- Melihat contoh-contoh sampel teks
- Mengecek panjang kalimat rata-rata
- Word frequency distribution
- Insight untuk preprocessing & modeling

> Pastikan sel **Setup** di bawah telah dijalankan sebelum menjalankan sel lainnya.

In [ ]:
"""SETUP — Wajib dijalankan sekali."""
import os
import sys

# Deteksi lingkungan Colab
IN_COLAB = "google.colab" in str(getattr(__builtins__, "__import__")("sys").modules.keys()) or \
           ("google.colab" in sys.modules)

print(f"In Google Colab: {IN_COLAB}")
print(f"Working Dir : {os.getcwd()}")

# --- Klona repo bila diperlukan (agar impor internal berfungsi) ---
PROJECT_DIR = "collab-workspace"
if IN_COLAB:
    if PROJECT_DIR not in os.listdir():
        print("\n>>> Cloning collab-workspace...")
        os.system(
            "git clone "
            "https://github.com/febriyansyahresearch-lab/"
            f"{PROJECT_DIR}.git"
        )
    os.chdir(PROJECT_DIR)
else:
    # Fallback: naik hingga menemukan folder collab-workspace
    while True:
        cur = os.path.basename(os.getcwd())
        if cur == PROJECT_DIR or ".git" in os.listdir():
            break
        os.chdir("..")

sys.path.insert(0, os.getcwd())

print(f"\nFinal Working Dir : {os.getcwd()}")
assert os.path.exists("projects"), (
    "'projects/' tidak ditemukan. Pastikan anda berada di root collab-workspace."
)
print("[OK] Root collab-workspace terdeteksi.")

# --- Instal dependensi bila kurang ---
try:
    import sklearn  # noqa: F401
except ModuleNotFoundError:
    print("\n>>> Installing missing deps...")
    os.system("pip install -q -r requirements.txt")

print("\n[DONE] Setup selesai.")


In [ ]:
"""
Imports & Konfigurasi Plotting
=================================
Library wajib untuk eksplorasi data teks.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style plotting yang enak dilihat
plt.style.use("ggplot")
sns.set_context("talk")
pd.set_option("display.max_colwidth", None)

# Modul internal proyek (tersedia setelah SETUP sukses)
from collections import Counter
from projects.nlp.src.data import generate_texts

print("[OK] Imports berhasil dimuat.")


In [ ]:
"""
Load Dataset & Gambaran Umum
==============================
Bangkitkan dataset sintetik untuk demonstrasi EDA.
Pada penggunaan nyata, gantilah dengan loader dataset sungguhan.
"""

SEED = 1337
np.random.seed(SEED)

# Bangkitkan 600 dokumen teks (contoh)
df = generate_texts(600, seed=SEED)

print(f"Dimensi dataset : {df.shape}")
print("-" * 120)
print(df.sample(min(len(df), 10)))


## Distribusi Kelas (Labels)

Seberapa seimbang jumlah sampel per kategori?

In [ ]:
# Hitung frekuensi tiap label
counts = df["label"].value_counts()

ax = counts.plot(kind="bar", figsize=(10, 5),
                 color=["steelblue", "tomato"],
                 edgecolor="black")
ax.set_title("Distribusi Label Teks", fontweight="bold", pad=15)
ax.set_xlabel("Label")
ax.set_ylabel("Frekuensi")
ax.grid(axis="y", alpha=0.3)

# Anotasikan nilai di atas batang
for idx, val in enumerate(counts.values):
    ax.text(idx, val + 2, str(val), ha="center", va="bottom", fontweight="bold")

balance_ratio = min(counts.min(), len(counts)) / counts.sum()
print(f"Nilai minimum ratio kelas terhadap total: {balance_ratio:.2%}")
print("=> Rasio > 0.85 berarti relatif SEIMBANG ; rasio rendah artinya imbalance.")
plt.show()


## Panjang Kalimat per Dokumen

Panjang teks sering berkorelasi kuat dengan jenis tugas NLP.

In [ ]:
# Kolom helper: banyaknya karakter & kata per dokumen
doc_len_char = df["text"].apply(lambda t: len(str(t)))   # jumlah huruf/spasi
doc_len_word = df["text"].apply(lambda t: len(str(t).split()))  # jumlah kata

summary_df = pd.DataFrame({
    "char_count": doc_len_char,
    "word_count": doc_len_word,
})
print(summary_df.describe().round(2))

# Histogram dua-duanya berdampingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(doc_len_char, bins=30, color="cornflowerblue", edgecolor="black")
axes[0].set_title("Distribusi Panjang Karakter", fontweight="bold")
axes[0].set_xlabel("#Karakter")
axes[0].grid(alpha=0.3)

axes[1].hist(doc_len_word, bins=30, color="lightcoral", edgecolor="black")
axes[1].set_title("Distribusi Jumlah Kata", fontweight="bold")
axes[1].set_xlabel("#Kata")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Top Words (Word Frequency)

Kata-kata apa yang paling dominan muncul di keseluruhan korpus?

In [ ]:
# Gabungkan SEMUA teks jadi satu blok besar, pecah per kata
words_all = []
for txt in df["text"].tolist():
    words_all.extend(str(txt).lower().split())

counter = Counter(words_all)
top_words = counter.most_common(20)

wc_top = dict(top_words)
palette_color = sns.color_palette("viridis", len(wc_top))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(list(wc_top.keys()), list(wc_top.values()),
              color=palette_color, edgecolor="black")
ax.set_title("20 Kata Paling Sering Muncul", fontweight="bold", pad=15)
ax.set_xlabel("Kata")
ax.set_ylabel("Frekuensi")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)

for rect, freq in zip(bars, wc_top.values()):
    height = rect.get_height()
    ax.annotate(freq, xy=(rect.get_x()+rect.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


## Contoh Sampel per Kelas

Lihatlah bentuk aktual teks untuk tiap kategori.

In [ ]:
unique_labels = sorted(df["label"].unique())
sample_rows = []

for lbl in unique_labels:
    tmp = df.loc[df["label"] == lbl]["text"]
    sample_rows.append({"label": lbl, "jumlah_sample": len(tmp)})

sampling_table = pd.concat([
    df.groupby("label")["text"].first(),
], axis=1)
sampling_table.columns = ["Contoh_Pertama"]

with pd.option_context("display.width", 150):
    display(sampling_table.reset_index())


## Kesimpulan & Rekomendasi Modeling

Ringkas temuan EDA dan tentukan strategi praproses/modeling.

In [ ]:
# ---- Auto-ringkasan sederhana berbasis angka ----
avg_w = float(np.mean([len(x.split()) for x in df["text"]]))
num_classes = df["label"].nunique()
most_label = counts.idxmax()

rec_lines = [
    "- Gunakan TF-IDF Vectorizer (batasi vocab mis. 5.000–10.000 term)",
    "- Stop-word removal sangat direkomendasikan utk mengurangi noise.",
    "- Mulailah baseline dengan Naive Bayes / Logistic Regression.",
]

print("========== REKAP HASIL EDA ==========")
print(f"- Jumlah dokumen      : {len(df)}")
print(f"- Jumlah kelas        : {num_classes}")
print(f"- Rata-rata kata/doc  : {avg_w:.1f}")
print(f"- Kelas mayoritas     : {most_label} ({counts.max()} sampel)")
print(f"- Balance ratio       : {float(counts.min())/sum(counts):.2%}")
print("\n---------- SARAN MODEL ----------")
for ln in rec_lines:
    print(ln)
print("------------------------------------")
